# 06 — Controlled Prospective Pilot

This notebook implements the prospective pilot phase after the retrospective modelling project. It freezes the final all-12 pre-call Logistic Regression as the primary ranking policy, keeps HistGradientBoosting as an optional challenger, forbids `duration` at scoring time, randomises eligible customers into equal-capacity policy pools, and defines the prospective outcome/economic analysis.

It does **not fabricate live results**. If `pilot_eligible_customers.csv` is absent, the notebook prepares the frozen models and pilot functions, then reports that live input is still required.

## 1. Frozen pilot specification

Model selection is over. Hyperparameters and features are fixed before pilot outcomes exist. The selected models may be refit on all historical rows for live scoring, but they are not retuned after the pilot begins.

In [1]:
from pathlib import Path
import warnings
import hashlib
import numpy as np
import pandas as pd
from scipy.stats import norm
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier

warnings.filterwarnings("ignore", category=FutureWarning)

SEED = 42
PILOT_SEED = 20260827
HISTORICAL_SOURCE = "term-deposit-marketing-2020-labelled.csv"
PILOT_INPUT = "pilot_eligible_customers.csv"

NUM = ["age","balance","day","campaign"]
CAT = ["job","marital","education","default","housing","loan","contact","month"]
PRE = NUM + CAT

LR_PARAMS = dict(C=8.483428982440726, penalty="l2", class_weight=None,
                 solver="liblinear", max_iter=3000, random_state=SEED)
HGB_PARAMS = dict(max_iter=200, max_depth=8, learning_rate=0.05,
                  max_leaf_nodes=15, l2_regularization=0.0,
                  min_samples_leaf=20, class_weight=None,
                  early_stopping=True, random_state=SEED)

assert "duration" not in PRE
print("Primary policy: all-12 unweighted Logistic Regression")
print("Live input present:", Path(PILOT_INPUT).exists())

Primary policy: all-12 unweighted Logistic Regression
Live input present: False


## 2. Validate historical training data and refit frozen models

The target helper is recreated from `y`. The frozen pilot models are then refit on all 40,000 historical rows, using only the 12 valid pre-call predictors.

In [2]:
historical = pd.read_csv(HISTORICAL_SOURCE)
if "y_binary" in historical.columns:
    historical = historical.drop(columns="y_binary")
historical["y_binary"] = historical["y"].eq("yes").astype(int)

assert len(historical) == 40000
assert int(historical["y_binary"].sum()) == 2896
assert set(PRE + ["y"]).issubset(historical.columns)

def prep():
    return ColumnTransformer([
        ("num", StandardScaler(), NUM),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT)
    ])

def build_lr():
    return Pipeline([("prep", prep()), ("model", LogisticRegression(**LR_PARAMS))])

def build_hgb():
    return Pipeline([("prep", prep()), ("model", HistGradientBoostingClassifier(**HGB_PARAMS))])

lr_pilot_model = build_lr().fit(historical[PRE], historical["y_binary"])
hgb_pilot_model = build_hgb().fit(historical[PRE], historical["y_binary"])

print("Frozen models refit on all historical rows.")
print("Historical base rate:", round(historical["y_binary"].mean(), 6))

Frozen models refit on all historical rows.
Historical base rate: 0.0724


## 3. Live input schema and leakage guard

The live file must contain a unique `customer_id` plus the 12 pre-call predictors. It must already contain only customers who pass the bank's eligibility, consent, suppression and contact-frequency rules. Outcome/post-call columns are rejected during assignment.

In [3]:
REQUIRED_LIVE = ["customer_id"] + PRE
FORBIDDEN_AT_ASSIGNMENT = {"duration","y","y_binary","subscribed"}

def validate_live_input(live):
    missing = set(REQUIRED_LIVE) - set(live.columns)
    if missing:
        raise ValueError(f"Missing live columns: {sorted(missing)}")
    forbidden = FORBIDDEN_AT_ASSIGNMENT & set(live.columns)
    if forbidden:
        raise ValueError(f"Post-call/outcome columns present: {sorted(forbidden)}")
    if live["customer_id"].isna().any():
        raise ValueError("customer_id contains missing values")
    if live["customer_id"].duplicated().any():
        raise ValueError("customer_id must be unique")
    if len(live) < 2:
        raise ValueError("Need at least two eligible customers")
    return True

print("Required live columns:", REQUIRED_LIVE)

Required live columns: ['customer_id', 'age', 'balance', 'day', 'campaign', 'job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month']


## 4. Score customers and randomise policy pools

Scores are generated from pre-call information only. Broad LR score strata are used only to balance randomisation. After randomisation, each arm applies its own policy inside its own pool.

In [4]:
def stable_assignment_id(customer_id):
    raw = f"{PILOT_SEED}|{customer_id}".encode()
    return hashlib.sha256(raw).hexdigest()[:16]

def score_live(live):
    validate_live_input(live)
    d = live.copy()
    d["lr_score"] = lr_pilot_model.predict_proba(d[PRE])[:,1]
    d["hgb_score"] = hgb_pilot_model.predict_proba(d[PRE])[:,1]
    d["lr_rank_all"] = d["lr_score"].rank(method="first", ascending=False).astype(int)
    q = min(10, max(2, len(d)//50))
    d["score_stratum"] = pd.qcut(d["lr_score"].rank(method="first"),
                                 q=q, labels=False, duplicates="drop")
    d["pilot_assignment_id"] = d["customer_id"].map(stable_assignment_id)
    return d

def randomise_pools(scored, include_hgb=False):
    arms = ["LR_ranked","Control"] + (["HGB_ranked"] if include_hgb else [])
    rng = np.random.default_rng(PILOT_SEED)
    out = scored.copy()
    out["arm"] = None
    for _, idx in out.groupby("score_stratum").groups.items():
        idx = rng.permutation(np.array(list(idx)))
        for j, row_idx in enumerate(idx):
            out.at[row_idx, "arm"] = arms[j % len(arms)]
    return out

print("Scoring and randomisation functions ready.")

Scoring and randomisation functions ready.


## 5. Apply equal call capacity

`calls_per_arm` is a required operational input. Historical 5%, 10% and 20% call depths are not automatically reused as pilot quotas. The control uses a bank BAU priority column if supplied; otherwise it selects randomly.

In [5]:
def select_equal_capacity(randomised, calls_per_arm, bau_priority_column=None):
    if not isinstance(calls_per_arm, (int, np.integer)) or calls_per_arm <= 0:
        raise ValueError("calls_per_arm must be a positive integer")
    rng = np.random.default_rng(PILOT_SEED + 1)
    pieces = []
    for arm, g0 in randomised.groupby("arm", sort=False):
        g = g0.copy()
        if len(g) < calls_per_arm:
            raise ValueError(f"{arm} has only {len(g)} customers")
        if arm == "LR_ranked":
            g = g.sort_values(["lr_score","pilot_assignment_id"], ascending=[False,True])
        elif arm == "HGB_ranked":
            g = g.sort_values(["hgb_score","pilot_assignment_id"], ascending=[False,True])
        elif bau_priority_column is not None:
            if bau_priority_column not in g.columns:
                raise ValueError(f"Missing BAU column: {bau_priority_column}")
            g = g.sort_values([bau_priority_column,"pilot_assignment_id"], ascending=[False,True])
        else:
            g = g.assign(_rand=rng.random(len(g))).sort_values("_rand")
        g["assigned_call"] = False
        g.loc[g.index[:calls_per_arm], "assigned_call"] = True
        pieces.append(g.drop(columns="_rand", errors="ignore"))
    assignment = pd.concat(pieces).sort_index()
    called = assignment.groupby("arm")["assigned_call"].sum()
    assert called.nunique() == 1
    return assignment

def audit_assignment(assignment):
    if assignment["customer_id"].duplicated().any():
        raise AssertionError("Duplicate customer IDs")
    if FORBIDDEN_AT_ASSIGNMENT & set(assignment.columns):
        raise AssertionError("Leakage/outcome column present")
    called = assignment.groupby("arm")["assigned_call"].sum()
    assert called.nunique() == 1
    return assignment.groupby("arm").agg(
        pool_n=("customer_id","size"),
        assigned_calls=("assigned_call","sum"),
        mean_lr_score=("lr_score","mean"),
        median_lr_score=("lr_score","median")
    )

print("Equal-capacity policy functions ready.")

Equal-capacity policy functions ready.


## 6. Prospective outcome analysis

After the predefined pilot window, merge outcomes back to the frozen assignment. The primary analysis compares the customers actually assigned a call under each randomised policy pool. The headline measure is subscriptions per 1,000 assigned calls, with absolute rate difference and relative lift versus control.

In [6]:
def bootstrap_rate_difference(a, b, reps=5000, seed=PILOT_SEED+2):
    rng = np.random.default_rng(seed)
    diffs = np.empty(reps)
    for i in range(reps):
        diffs[i] = rng.choice(a, len(a), replace=True).mean() - rng.choice(b, len(b), replace=True).mean()
    return np.quantile(diffs, [0.025,0.975])

def analyse_pilot(d, control_arm="Control"):
    if "subscribed" not in d.columns:
        raise ValueError("Need subscribed=0/1 outcomes")
    called = d.loc[d["assigned_call"].eq(True)].copy()
    if called.empty:
        raise ValueError("No assigned calls found")
    if called["subscribed"].isna().any():
        raise ValueError("Primary analysis requires complete outcomes for assigned calls")
    if not set(called["subscribed"].unique()).issubset({0,1}):
        raise ValueError("subscribed must be 0/1")

    summary = called.groupby("arm").agg(
        assigned_calls=("customer_id","size"),
        subscriptions=("subscribed","sum"),
        subscription_rate=("subscribed","mean")
    )
    summary["subscriptions_per_1000_assigned_calls"] = 1000*summary["subscription_rate"]

    if control_arm not in summary.index:
        raise ValueError(f"Control arm {control_arm!r} not found")
    control = called.loc[called["arm"].eq(control_arm),"subscribed"].to_numpy(float)
    rows = []
    for arm in summary.index:
        if arm == control_arm:
            continue
        x = called.loc[called["arm"].eq(arm),"subscribed"].to_numpy(float)
        lo, hi = bootstrap_rate_difference(x, control)
        rows.append({
            "arm": arm,
            "control": control_arm,
            "absolute_rate_difference": x.mean()-control.mean(),
            "difference_ci_95_low": lo,
            "difference_ci_95_high": hi,
            "relative_lift": x.mean()/control.mean() if control.mean()>0 else np.nan
        })
    return summary, pd.DataFrame(rows)

print("Prospective outcome-analysis functions ready.")


Prospective outcome-analysis functions ready.


## 7. Sample-size planning and economics

Pilot size must come from the control conversion rate and the minimum worthwhile improvement. Economics are calculated only after the bank supplies actual call cost and net contribution per subscription.

In [7]:
def sample_size_per_arm(control_rate, target_rate, alpha=0.05, power=0.80):
    p1, p2 = float(control_rate), float(target_rate)
    if not (0 < p1 < 1 and 0 < p2 < 1) or p1 == p2:
        raise ValueError("Provide two distinct rates between 0 and 1")
    za = norm.ppf(1-alpha/2)
    zp = norm.ppf(power)
    pbar = (p1+p2)/2
    n = (za*np.sqrt(2*pbar*(1-pbar)) + zp*np.sqrt(p1*(1-p1)+p2*(1-p2)))**2/(p2-p1)**2
    return int(np.ceil(n))

def add_economics(summary, call_cost, net_contribution_per_subscription):
    if call_cost < 0 or net_contribution_per_subscription < 0:
        raise ValueError("Economic inputs must be non-negative")
    out = summary.copy()
    out["gross_contribution"] = out["subscriptions"]*float(net_contribution_per_subscription)
    out["calling_cost"] = out["assigned_calls"]*float(call_cost)
    out["net_value"] = out["gross_contribution"]-out["calling_cost"]
    return out

print("Planning reference only, using historical 7.24% control rate:")
for target in [0.08,0.09,0.10]:
    print(target, sample_size_per_arm(0.0724,target))

Planning reference only, using historical 7.24% control rate:
0.08 19131
0.09 3780
0.1 1623


## 8. Current launch status

A real prospective pilot cannot be created from the historical outcomes. The remaining launch inputs are a genuinely new eligible-customer snapshot, the equal call budget, the control-policy definition, and bank economics. No pilot outcome is invented while those inputs are absent.

In [8]:
if Path(PILOT_INPUT).exists():
    live = pd.read_csv(PILOT_INPUT)
    validate_live_input(live)
    scored_live = score_live(live)
    print(f"Live input validated: {len(scored_live):,} eligible customers.")
    print("Next: set calls_per_arm, randomise pools, freeze/export assignment before any outcomes are observed.")
else:
    print("READY / WAITING FOR LIVE INPUT: pilot_eligible_customers.csv is not present.")
    print("No prospective assignment or result has been fabricated.")

READY / WAITING FOR LIVE INPUT: pilot_eligible_customers.csv is not present.
No prospective assignment or result has been fabricated.
